In [11]:
import numpy as np
import matplotlib.pyplot as plt

# %matplotlib tk  # keep if you like interactive window in notebooks

# --- Geometry / Numerics ---
L = 0.20
N = 50
dx = L / N
dt = 0.01
steps = 500

# --- Material properties (stainless-ish) ---
alpha = 4.2e-6          # m^2/s  (thermal diffusivity)
rho = 8000              # kg/m^3
cp  = 500               # J/(kg*K)
thickness = 0.003       # m (3mm pot bottom)

# --- Convection to water above ---
h = 500                 # W/(m^2*K)
T_water = 70.0          # °C

# Convert convection coefficient into temperature decay rate (1/s)
k_conv = h / (rho * cp * thickness)  # 1/s

# --- Initialize temperature field ---
u = np.full((N, N), 20.0)

# Burner mask: circle in center
center = N // 2
radius = N // 4
Y, X = np.ogrid[:N, :N]
burner = (X - center)**2 + (Y - center)**2 <= radius**2

T_burner = 150.0
u[burner] = T_burner

# Stability check (explicit 2D)
r = alpha * dt / dx**2
if r > 0.25:
    print(f"Warning: explicit scheme may be unstable (r={r:.3f} > 0.25).")

def step(u):
    u_next = u.copy()

    # Laplacian on interior (vectorized)
    lap = (
        u[2:, 1:-1] + u[:-2, 1:-1] +
        u[1:-1, 2:] + u[1:-1, :-2] -
        4.0 * u[1:-1, 1:-1]
    )

    diffusion = r * lap

    # Convection: dT = -k_conv*(T - T_water)*dt
    cooling = -k_conv * (u[1:-1, 1:-1] - T_water) * dt

    u_next[1:-1, 1:-1] = u[1:-1, 1:-1] + diffusion + cooling

    # Boundary condition choice:
    # Option A (simple): hold edges at room temp (Dirichlet)
    # u_next[0, :] = u_next[-1, :] = u_next[:, 0] = u_next[:, -1] = 20.0

    # Option B (insulated edges): zero-gradient (Neumann)
    u_next[0, :]  = u_next[1, :]
    u_next[-1, :] = u_next[-2, :]
    u_next[:, 0]  = u_next[:, 1]
    u_next[:, -1] = u_next[:, -2]

    # Enforce burner held at constant temperature each step (Dirichlet burner)
    #u_next[burner] = T_burner
    # for power input conversion
    q_flux = 20_000  # W/m^2 (example)
    u_next[burner] += (q_flux / (rho * cp * thickness)) * dt

    return u_next

# --- Visualize ---
plt.ion()
fig, ax = plt.subplots()
im = ax.imshow(u, cmap="hot", extent=[0, L, 0, L], vmin=20, vmax=150)
plt.colorbar(im, ax=ax, label="Temp (°C)")

for t in range(steps):
    u = step(u)
    if t % 10 == 0:
        im.set_data(u)
        ax.set_title(f"Pot Bottom Temperature at t = {t*dt:.2f}s")
        plt.pause(0.001)

plt.ioff()
plt.show()

In [16]:
import numpy as np
import matplotlib.pyplot as plt

# %matplotlib tk   # uncomment if you want a separate interactive window (not always supported everywhere)

# ============================================================
# 2D POT-BOTTOM HEAT MAP  (stovetop / metal bottom)
# + 1D "VERTICAL" POT MODEL (water bulk + headspace + lid vent)
# ============================================================

# ---------------------------
# A) Geometry / Numerics
# ---------------------------
L = 0.20          # pot diameter footprint modeled (m) ~ 20 cm
N = 60            # grid resolution
dx = L / N
dt = 0.01         # seconds
steps = 4000      # total simulation steps
t_total = steps * dt

# Explicit diffusion stability sanity (for 2D)
alpha_steel = 4.2e-6  # m^2/s
r = alpha_steel * dt / dx**2
print(f"2D explicit stability parameter r = {r:.4f} (want <= ~0.25)")

# Burner definition (circle at center)
center = N // 2
radius = N // 4
Y, X = np.ogrid[:N, :N]
burner = (X - center)**2 + (Y - center)**2 <= radius**2
T_burner = 180.0  # °C (metal boundary on burner region)

# Initial pot-bottom temp field
u = np.full((N, N), 20.0)  # °C
u[burner] = T_burner

# Boundary condition for edges of the 2D metal plate:
# "insulated" (zero-gradient)
def apply_neumann_edges(arr):
    arr[0, :]  = arr[1, :]
    arr[-1, :] = arr[-2, :]
    arr[:, 0]  = arr[:, 1]
    arr[:, -1] = arr[:, -2]
    return arr

def step_metal(u):
    u_next = u.copy()

    # Laplacian interior (vectorized)
    lap = (
        u[2:, 1:-1] + u[:-2, 1:-1] +
        u[1:-1, 2:] + u[1:-1, :-2] -
        4.0 * u[1:-1, 1:-1]
    )
    u_next[1:-1, 1:-1] = u[1:-1, 1:-1] + r * lap

    # edges: insulated
    u_next = apply_neumann_edges(u_next)

    # enforce burner as fixed temperature each step (Dirichlet burner)
    u_next[burner] = T_burner
    return u_next

# ---------------------------
# B) Pot / Water / Headspace Model (vertical projection)
# ---------------------------
# Pot dimensions (reasonable defaults)
R_pot = 0.10             # m (radius ~10 cm for a 20 cm diameter pot)
H_pot = 0.12             # m (12 cm tall pot)
A_surf = np.pi * R_pot**2

fill_frac = 1/3          # "1/3 full at start"
rho_w = 997.0            # kg/m^3
cp_w = 4180.0            # J/(kg*K)

T_amb = 20.0             # °C
P_amb = 101325.0         # Pa
R_gas = 8.314462618      # J/(mol*K)
M_w = 0.01801528         # kg/mol
h_fg = 2.256e6           # J/kg (latent heat, approx near 100C)

# Initial water inventory
V_w0 = A_surf * (H_pot * fill_frac)
m_w = rho_w * V_w0       # kg
T_w = T_amb              # °C

# Headspace volume (dynamic, from current water volume)
def headspace_volume(m_w):
    V_w = m_w / rho_w
    V_total = A_surf * H_pot
    V_g = max(V_total - V_w, 1e-6)
    return V_g

# Initial headspace: assume at ambient pressure with dry-ish air + some initial vapor at room temp
# We'll start with 50% RH as a default (you can change)
RH0 = 0.50

def psat_water_pa(Tc):
    """
    Saturation vapor pressure of water in Pa.
    Uses Antoine (valid ~1-100C). For >100C it will be approximate but still behaves.
    """
    # Antoine constants for water (1–100C), P in mmHg
    A = 8.07131
    B = 1730.63
    C = 233.426
    P_mmHg = 10 ** (A - B / (C + Tc))
    return P_mmHg * 133.322368  # Pa

def clamp(x, lo, hi):
    return max(lo, min(hi, x))

# Initialize gas moles
Tg = T_w + 273.15
Vg = headspace_volume(m_w)
Pv0 = RH0 * psat_water_pa(T_w)
nv = Pv0 * Vg / (R_gas * Tg)                        # moles of water vapor
na = (P_amb - Pv0) * Vg / (R_gas * Tg)              # moles of dry air

# Coupling from 2D metal field -> water heat input
h_mw = 2500.0   # W/(m^2*K) metal->water effective HTC (tune)
# Optional heat losses from water to ambient through walls/lid (very simple lumped)
UA_loss = 8.0   # W/K (tune; set 0 to disable)

# Evaporation kinetics when lid is ON (moles/s = K_evap*A*(Psat - Pv))
# Units: K_evap [mol/(s*m^2*Pa)]
K_evap = 2e-7   # tune (bigger -> faster humidity equilibration/evap)

# Lid/vent schedule
t_lid_on = 0.0            # lid starts on immediately (change if desired)
t_vent_start = 40.0       # seconds
vent_duration = 1.0       # seconds (0.1–3 s typical)
t_heat_off = 120.0        # seconds: shut burner off

# Vent dynamics (first-order relaxation of total moles to ambient)
tau_vent = 0.35           # seconds (tune to make venting "stronger/weaker")

def lid_state(t):
    # Lid ON except during vent window
    if t < t_lid_on:
        return False
    if t_vent_start <= t <= t_vent_start + vent_duration:
        return False
    return True

# For convenience, we’ll turn the burner off by reducing T_burner after t_heat_off
def burner_temp_schedule(t):
    return 180.0 if t < t_heat_off else 20.0

# ---------------------------
# C) Run + visualize
# ---------------------------
plt.ion()

fig = plt.figure(figsize=(12, 5))
ax0 = plt.subplot2grid((1, 2), (0, 0))
ax1 = plt.subplot2grid((1, 2), (0, 1))

im = ax0.imshow(u, cmap="hot", vmin=20, vmax=200, extent=[0, L, 0, L])
cb = plt.colorbar(im, ax=ax0, label="Pot-bottom Temp (°C)")
ax0.set_title("2D Pot-Bottom Temperature")

# Timeseries storage
ts = []
Tw_s = []
mw_s = []
P_s = []
Pv_s = []
mdot_evap_s = []
lid_s = []

line1, = ax1.plot([], [], label="Water Temp (°C)")
line2, = ax1.plot([], [], label="Headspace P - Pamb (kPa)")
line3, = ax1.plot([], [], label="Evap m_dot (g/s)")
ax1.set_xlabel("Time (s)")
ax1.grid(True)
ax1.legend(loc="upper left")

def update_timeseries_plot():
    if len(ts) < 2:
        return
    ax1.cla()
    ax1.grid(True)

    ax1.plot(ts, Tw_s, label="Water Temp (°C)")

    # pressure delta in kPa
    dP_kPa = (np.array(P_s) - P_amb) / 1000.0
    ax1.plot(ts, dP_kPa, label="Headspace ΔP (kPa)")

    # evaporation g/s
    ax1.plot(ts, np.array(mdot_evap_s) * 1000.0, label="Evap m_dot (g/s)")

    ax1.set_xlabel("Time (s)")
    ax1.legend(loc="upper left")

# Main loop
for k in range(steps):
    t = k * dt

    # burner schedule
    T_burner = burner_temp_schedule(t)

    # step metal plate
    u = step_metal(u)

    # compute heat input from metal to water (using the footprint area of the pot)
    # Here we assume the 2D grid already represents the pot-bottom area.
    # Q_in = ∫ h_mw (T_metal - T_water) dA
    dA = (L / N) ** 2
    Q_in = h_mw * np.sum((u - T_w)) * dA  # W  (can be negative if water hotter than metal)

    # simple ambient losses from water
    Q_loss = UA_loss * (T_w - T_amb)      # W

    # Headspace update setup
    Vg = headspace_volume(m_w)
    Tg = (T_w + 273.15)                  # K (simple: headspace at water temp)
    Psat = psat_water_pa(T_w)
    Pv = (nv * R_gas * Tg) / Vg          # Pa

    # Condensation clamp (enforce Pv <= Psat)
    if Pv > Psat:
        nv_sat = Psat * Vg / (R_gas * Tg)
        nv = nv_sat
        Pv = Psat

    # Evaporation rate (depends on lid)
    lid_on = lid_state(t)

    if lid_on:
        # kinetic drive toward saturation in headspace
        dn_evap = K_evap * A_surf * max(Psat - Pv, 0.0)  # mol/s
    else:
        # lid off: allow faster evaporation to ambient humidity (use RH0 as ambient)
        Pv_amb = RH0 * psat_water_pa(T_amb)
        dn_evap = K_evap * 8.0 * A_surf * max(Psat - Pv_amb, 0.0)  # faster when open

    # convert to mass flow, cap by remaining water
    mdot_evap = dn_evap * M_w            # kg/s
    mdot_evap = min(mdot_evap, m_w / max(dt, 1e-9) * 0.1)  # avoid draining in one step

    # Energy balance on water (bulk mixed)
    # m cp dT/dt = Q_in - mdot*h_fg - Q_loss
    dTdt = (Q_in - mdot_evap * h_fg - Q_loss) / max(m_w * cp_w, 1e-9)
    T_w = T_w + dTdt * dt
    T_w = max(T_w, -10.0)  # keep sane

    # Update water mass
    m_w = m_w - mdot_evap * dt
    m_w = max(m_w, 0.0)

    # Add vapor moles from evaporation
    nv = nv + dn_evap * dt

    # Venting event: when lid off, relax total moles toward ambient pressure
    if not lid_on:
        n_tot = na + nv
        n_eq = P_amb * Vg / (R_gas * Tg)
        dn_tot_dt = -(n_tot - n_eq) / tau_vent  # mol/s
        dn_tot = dn_tot_dt * dt

        # remove proportionally from air+vapor (or add if below ambient)
        if n_tot > 1e-12:
            frac_v = nv / n_tot
        else:
            frac_v = 0.0

        nv = nv + frac_v * dn_tot
        na = na + (1.0 - frac_v) * dn_tot

        # keep non-negative
        nv = max(nv, 0.0)
        na = max(na, 0.0)

        # Re-clamp saturation after vent (condensation)
        Pv = (nv * R_gas * Tg) / Vg
        Psat = psat_water_pa(T_w)
        if Pv > Psat:
            nv = Psat * Vg / (R_gas * Tg)

    # Compute pressures for logging
    P = (na + nv) * R_gas * Tg / Vg
    Pv = (nv * R_gas * Tg) / Vg

    # Log every few steps (for speed)
    if k % 5 == 0:
        ts.append(t)
        Tw_s.append(T_w)
        mw_s.append(m_w)
        P_s.append(P)
        Pv_s.append(Pv)
        mdot_evap_s.append(mdot_evap)
        lid_s.append(1 if lid_on else 0)

    # Update plots occasionally
    if k % 50 == 0:
        im.set_data(u)
        ax0.set_title(
            f"2D Pot-Bottom Temp | t={t:6.2f}s | Lid={'ON' if lid_on else 'OFF'} | Burner={'ON' if t<t_heat_off else 'OFF'}"
        )
        update_timeseries_plot()
        plt.pause(0.001)

plt.ioff()
plt.show()

# Quick summary prints
print("\n--- Summary ---")
print(f"Final water temp: {T_w:.2f} °C")
print(f"Final water mass: {m_w:.3f} kg")
print(f"Final headspace pressure: {P:.0f} Pa  (ΔP={(P-P_amb)/1000:.2f} kPa)")
print(f"Final water vapor partial pressure: {Pv:.0f} Pa")

2D explicit stability parameter r = 0.0038 (want <= ~0.25)

--- Summary ---
Final water temp: 43.02 °C
Final water mass: 1.253 kg
Final headspace pressure: 110199 Pa  (ΔP=8.87 kPa)
Final water vapor partial pressure: 2177 Pa


In [15]:
import numpy as np
import matplotlib.pyplot as plt

# OPTION 2 for HeatSIM
# %matplotlib tk  # uncomment if you're in a notebook and want a pop-out interactive window

# =============================================================================
# 2D POT BOTTOM (stovetop) TEMPERATURE FIELD  +  "VERTICAL" POT MODEL
# - 2D bottom diffusion model stays independent of water for now
# - 1D (bulk) water temperature + headspace vapor/air + lid vent events
# - Outputs: evaporation mass flow rate, internal pressure vs ambient, water temp
# =============================================================================

# -------------------------
# 0) Physical / Geometry
# -------------------------
# 2D bottom geometry (matches your earlier setup: 20cm diameter region)
L = 0.20          # m (20 cm)
N = 60            # grid
dx = L / N
dA = dx * dx

dt = 0.01         # s
steps = 2000      # total simulation steps (20 s)

# Bottom plate diffusion (treat this as a "stovetop/pot-bottom" temperature sheet)
alpha_steel = 4.2e-6  # m^2/s

# Bottom-to-ambient cooling (keeps bottom from staying crazy-hot forever)
T_amb = 20.0      # °C
k_bottom_cool = 0.08  # 1/s  (tune: larger = bottom cools faster to ambient)

# Burner settings (simple: hold burner region at fixed temperature while heat_on)
T_burner = 150.0  # °C
heat_on = True

# Pot geometry (vertical projection model)
R_pot = 0.10      # m (10 cm radius -> ~20 cm diameter)
H_pot = 0.12      # m (12 cm inner height)
A_pot = np.pi * R_pot**2

fill_frac = 1/3   # 1/3 full water
rho_w = 1000.0    # kg/m^3
cp_w = 4180.0     # J/(kg*K)
Mw = 0.01801528   # kg/mol (water)
Rgas = 8.314462618 # J/(mol*K)

# Water latent heat (rough constant; good enough for now)
h_fg = 2.26e6     # J/kg

# Bottom metal -> water heat transfer coefficient (tunable)
h_mw = 2500.0     # W/(m^2*K)  (try 500..10000)

# Pot heat losses to ambient (optional; keep modest)
UA_loss = 3.0     # W/K  (try 0..10)

# Headspace initial ambient conditions
P_amb = 101325.0  # Pa
RH_amb = 0.50     # relative humidity (0..1)

# Lid vent schedule
# Lid ON most of the time; remove briefly between t_open_start and t_open_end
t_open_start = 6.0  # s
t_open_end   = 7.5  # s   (0.1..3s window; here 1.5s)

# Heat shutoff time (stops enforcing burner temperature)
t_heat_off = 10.0   # s

# Vent "time constant" when lid is off (how fast pressure relaxes to ambient)
tau_vent = 0.25      # s (smaller -> faster blowdown)

# Evaporation kinetics constants (simple pressure-driving forms)
# Units: mol / (s*m^2*Pa)
K_evap_sealed = 5.0e-8
K_evap_open   = 2.0e-7

# -------------------------
# 1) Helper: water saturation vapor pressure (Antoine) -> Pa
# Valid-ish for ~1–100°C (good enough for this sim)
# -------------------------
def psat_water_Pa(T_C: float) -> float:
    # Antoine constants for water (T in °C, P in mmHg)
    A, B, C = 8.07131, 1730.63, 233.426
    # guard
    T_C = float(np.clip(T_C, 0.0, 100.0))
    P_mmHg = 10 ** (A - (B / (C + T_C)))
    return P_mmHg * 133.322368  # mmHg -> Pa

# -------------------------
# 2) Initialize 2D bottom temperature field u(x,y)
# -------------------------
u = np.full((N, N), T_amb, dtype=float)

cy = cx = N // 2
radius = N // 4
Y, X = np.ogrid[:N, :N]
burner_mask = (X - cx)**2 + (Y - cy)**2 <= radius**2

# Start with a hot burner patch
u[burner_mask] = T_burner

# Explicit 2D diffusion stability factor
r = alpha_steel * dt / dx**2
if r > 0.25:
    print(f"WARNING: 2D explicit diffusion may be unstable: r={r:.3f} (>0.25). Reduce dt or increase N.")

def step_bottom(u, heat_on_flag=True):
    """Vectorized explicit diffusion + mild cooling to ambient; burner held if heat_on."""
    u_next = u.copy()

    # Laplacian on interior
    lap = (
        u[2:, 1:-1] + u[:-2, 1:-1] +
        u[1:-1, 2:] + u[1:-1, :-2] -
        4.0 * u[1:-1, 1:-1]
    )
    u_next[1:-1, 1:-1] = u[1:-1, 1:-1] + r * lap

    # Insulated edges (zero gradient)
    u_next[0, :]  = u_next[1, :]
    u_next[-1, :] = u_next[-2, :]
    u_next[:, 0]  = u_next[:, 1]
    u_next[:, -1] = u_next[:, -2]

    # Bottom cools toward ambient (keeps it realistic-ish)
    u_next += -k_bottom_cool * (u_next - T_amb) * dt

    # Burner enforcement
    if heat_on_flag:
        u_next[burner_mask] = T_burner

    return u_next

# -------------------------
# 3) Initialize vertical pot model state
# -------------------------
# initial water volume/mass (1/3 full)
V_water0 = A_pot * (H_pot * fill_frac)
m_w = rho_w * V_water0
T_w = T_amb  # °C, initial

# headspace volume
def headspace_volume(m_w_kg: float) -> float:
    h_w = m_w_kg / (rho_w * A_pot)
    h_w = np.clip(h_w, 0.0, H_pot * 0.999)  # avoid zero headspace
    return A_pot * (H_pot - h_w)

V_g = headspace_volume(m_w)

# initial headspace temperature (first approximation)
T_g = T_w  # °C

# initial vapor partial pressure from ambient humidity
Pv0 = RH_amb * psat_water_Pa(T_amb)
Pa0 = P_amb - Pv0

T0K = T_g + 273.15
n_v = Pv0 * V_g / (Rgas * T0K)
n_a = Pa0 * V_g / (Rgas * T0K)

def pressure_pa(n_a, n_v, V_g, T_g_C):
    return (n_a + n_v) * Rgas * (T_g_C + 273.15) / V_g

# -------------------------
# 4) Storage for plots
# -------------------------
ts = np.zeros(steps)
Tw_hist = np.zeros(steps)
P_hist = np.zeros(steps)
mdot_hist = np.zeros(steps)
mliq_hist = np.zeros(steps)
bottom_mean_hist = np.zeros(steps)

# -------------------------
# 5) Visualization setup
# -------------------------
plt.ion()

fig = plt.figure(figsize=(11, 7))
gs = fig.add_gridspec(2, 2)

ax_map = fig.add_subplot(gs[:, 0])
im = ax_map.imshow(u, cmap="hot", extent=[0, L, 0, L], vmin=T_amb, vmax=T_burner)
cb = fig.colorbar(im, ax=ax_map, label="Bottom Temp (°C)")
ax_map.set_title("2D Pot Bottom Temperature")
ax_map.set_xlabel("x (m)")
ax_map.set_ylabel("y (m)")

ax_T = fig.add_subplot(gs[0, 1])
ax_P = fig.add_subplot(gs[1, 1])

line_T, = ax_T.plot([], [], lw=2)
ax_T.set_title("Water Temperature")
ax_T.set_xlabel("time (s)")
ax_T.set_ylabel("T_w (°C)")
ax_T.grid(True)

line_P, = ax_P.plot([], [], lw=2, color = 'green')
ax_P.set_title("Pressure Differential & Evaporation")
ax_P.set_xlabel("time (s)")
ax_P.set_ylabel("P_in - P_amb (Pa)")
ax_P.grid(True)

# second axis for mdot
ax_mdot = ax_P.twinx()
line_mdot, = ax_mdot.plot([], [], lw=2, linestyle="--")
ax_mdot.set_ylabel("m_dot_evap (g/s)")

# -------------------------
# 6) Main simulation
# -------------------------
for k in range(steps):
    t = k * dt
    ts[k] = t

    # Schedule: heat on/off
    heat_on = (t < t_heat_off)

    # Schedule: lid on/off
    lid_on = not (t_open_start <= t <= t_open_end)

    # Step 2D bottom model (independent)
    u = step_bottom(u, heat_on_flag=heat_on)

    # ---------
    # Coupling: compute heat flow into water from bottom temperature field
    # ---------
    # Bottom-to-water heat transfer across the pot footprint.
    # Your u-field is a 20cm square area; we’ll treat it as the pot bottom area.
    # Q_in = ∑ h_mw * (T_bottom - T_w) dA   (only where T_bottom > T_w contributes net heat)
    dT = u - T_w
    Q_in = np.sum(h_mw * dT * dA)

    # Optional: do not allow "reverse heating" (water heating bottom) if you want a one-way model
    # Uncomment if desired:
    # Q_in = max(Q_in, 0.0)

    # ---------
    # Headspace volume depends on remaining water mass
    # ---------
    V_g = headspace_volume(m_w)
    T_g = T_w  # simple assumption for now (gas tracks water temp)

    # Compute current pressures / partials
    P_in = pressure_pa(n_a, n_v, V_g, T_g)
    Pv = n_v * Rgas * (T_g + 273.15) / V_g
    Psat = psat_water_Pa(T_w)

    # ---------
    # Lid OFF: vent/blowdown toward ambient pressure with time constant tau_vent
    # ---------
    if not lid_on:
        T_K = T_g + 273.15
        n_tot = n_a + n_v
        n_eq = P_amb * V_g / (Rgas * T_K)

        dn_tot = -(n_tot - n_eq) * (dt / tau_vent)

        # Remove proportionally to composition
        if n_tot > 1e-12:
            n_v += (n_v / n_tot) * dn_tot
            n_a += (n_a / n_tot) * dn_tot

        # Clamp to nonnegative
        n_v = max(n_v, 0.0)
        n_a = max(n_a, 0.0)

    # Recompute Pv after venting step
    Pv = n_v * Rgas * (T_g + 273.15) / V_g

    # ---------
    # Evaporation model
    # - sealed: driven by (Psat - Pv)
    # - open: driven by (Psat - Pv_amb)
    # ---------
    if lid_on:
        drive = max(Psat - Pv, 0.0)
        dn_evap = K_evap_sealed * A_pot * drive * dt  # mol
    else:
        Pv_amb = RH_amb * psat_water_Pa(T_amb)
        drive = max(Psat - Pv_amb, 0.0)
        dn_evap = K_evap_open * A_pot * drive * dt    # mol

    # Convert to mass, limited by remaining liquid
    dm_evap = dn_evap * Mw
    dm_evap = min(dm_evap, m_w)  # can't evaporate more than liquid exists

    # Update masses/moles
    m_w -= dm_evap
    n_v += (dm_evap / Mw)  # add evaporated vapor moles to headspace

    # ---------
    # Condensation clamp: enforce Pv <= Psat (if cooling makes it supersaturated)
    # ---------
    T_K = T_g + 273.15
    n_v_max = Psat * V_g / (Rgas * T_K)
    if n_v > n_v_max:
        # Excess condenses back to liquid
        dn_cond = n_v - n_v_max
        n_v = n_v_max
        m_w += dn_cond * Mw  # return to liquid

    # ---------
    # Water temperature energy balance
    # ---------
    # Losses to ambient through walls/lid (simple UA model)
    Q_loss = UA_loss * (T_w - T_amb)

    # Latent power from evaporation this step (J/s approx)
    mdot_evap = (dm_evap / dt) if dt > 0 else 0.0
    Q_lat = mdot_evap * h_fg

    # dT = (Q_in - Q_loss - Q_lat) * dt / (m_w * cp)
    # Guard against near-zero mass
    if m_w > 1e-6:
        dT_w = (Q_in - Q_loss - Q_lat) * dt / (m_w * cp_w)
        T_w += dT_w

    # ---------
    # Record
    # ---------
    P_in = pressure_pa(n_a, n_v, V_g, T_g)
    Tw_hist[k] = T_w
    P_hist[k] = P_in - P_amb
    mdot_hist[k] = mdot_evap * 1000.0  # g/s
    mliq_hist[k] = m_w
    bottom_mean_hist[k] = float(np.mean(u))

    # ---------
    # Update plots occasionally
    # ---------
    if k % 10 == 0:
        im.set_data(u)
        ax_map.set_title(
            f"2D Bottom Temp | t={t:5.2f}s | heat_on={heat_on} | lid_on={lid_on}"
        )

        line_T.set_data(ts[:k+1], Tw_hist[:k+1])
        ax_T.relim(); ax_T.autoscale_view()

        line_P.set_data(ts[:k+1], P_hist[:k+1])
        line_mdot.set_data(ts[:k+1], mdot_hist[:k+1])
        ax_P.relim(); ax_P.autoscale_view()
        ax_mdot.relim(); ax_mdot.autoscale_view()

        plt.pause(0.001)

plt.ioff()
plt.show()

print("\n--- Summary ---")
print(f"Final water temp: {T_w:.2f} °C")
print(f"Final liquid water mass: {m_w:.3f} kg")
print(f"Final pressure delta (P_in - P_amb): {P_hist[-1]:.1f} Pa")



--- Summary ---
Final water temp: 27.71 °C
Final liquid water mass: 1.257 kg
Final pressure delta (P_in - P_amb): 1426.3 Pa


In [17]:
import numpy as np
import matplotlib.pyplot as plt
from matplotlib.widgets import Slider, Button

# ============================================================
# Interactive: 2D pot-bottom heat diffusion + 1D vertical pot model
# Sliders ("twist knobs") + Power button
# ============================================================

# ---------------------------
# A) Geometry / Numerics
# ---------------------------
L = 0.20
N = 60
dx = L / N
dt = 0.01

alpha_steel = 4.2e-6  # m^2/s
r = alpha_steel * dt / dx**2
print(f"2D explicit stability parameter r = {r:.4f} (want <= ~0.25)")

center = N // 2
radius = N // 4
Y, X = np.ogrid[:N, :N]
burner = (X - center)**2 + (Y - center)**2 <= radius**2

def apply_neumann_edges(arr):
    arr[0, :]  = arr[1, :]
    arr[-1, :] = arr[-2, :]
    arr[:, 0]  = arr[:, 1]
    arr[:, -1] = arr[:, -2]
    return arr

def step_metal(u, T_burner):
    u_next = u.copy()
    lap = (
        u[2:, 1:-1] + u[:-2, 1:-1] +
        u[1:-1, 2:] + u[1:-1, :-2] -
        4.0 * u[1:-1, 1:-1]
    )
    u_next[1:-1, 1:-1] = u[1:-1, 1:-1] + r * lap
    u_next = apply_neumann_edges(u_next)
    u_next[burner] = T_burner
    return u_next

# ---------------------------
# B) Pot / Water / Headspace model
# ---------------------------
R_pot = 0.10
H_pot = 0.12
A_surf = np.pi * R_pot**2

fill_frac = 1/3
rho_w = 997.0
cp_w = 4180.0
T_amb = 20.0
P_amb = 101325.0
R_gas = 8.314462618
M_w = 0.01801528
h_fg = 2.256e6

def psat_water_pa(Tc):
    # Antoine (1–100C). Still behaves reasonably outside, but treat as approximate.
    A = 8.07131
    B = 1730.63
    C = 233.426
    P_mmHg = 10 ** (A - B / (C + Tc))
    return P_mmHg * 133.322368

def headspace_volume(m_w):
    V_w = m_w / rho_w
    V_total = A_surf * H_pot
    return max(V_total - V_w, 1e-6)

# ---------------------------
# C) Initial state
# ---------------------------
u = np.full((N, N), T_amb)

V_w0 = A_surf * (H_pot * fill_frac)
m_w0 = rho_w * V_w0
T_w0 = T_amb

RH0 = 0.50
Vg0 = headspace_volume(m_w0)
Tg0 = T_w0 + 273.15
Pv0 = RH0 * psat_water_pa(T_w0)
nv0 = Pv0 * Vg0 / (R_gas * Tg0)
na0 = (P_amb - Pv0) * Vg0 / (R_gas * Tg0)

# ---------------------------
# D) Controls (defaults)
# ---------------------------
state = {
    "power_on": True,
    "lid_on": True,
    "T_burner_on": 180.0,   # °C when power is ON
    "T_burner_off": T_amb,  # °C when power is OFF
    "h_mw": 2500.0,         # W/m^2/K metal->water
    "K_evap": 2e-7,         # mol/(s*m^2*Pa)
    "tau_vent": 0.35,       # s
    "UA_loss": 8.0,         # W/K
    "open_evap_mult": 8.0,  # faster evaporation when lid OFF
    "RH_amb": RH0,          # ambient RH used when lid OFF
}

# ---------------------------
# E) Simulation step for vertical model
# ---------------------------
def vertical_step(u, T_w, m_w, nv, na, params):
    # Burner temp based on power
    T_burner = params["T_burner_on"] if params["power_on"] else params["T_burner_off"]

    # Step the metal
    u = step_metal(u, T_burner)

    # Heat input from metal to water
    dA = (L / N) ** 2
    Q_in = params["h_mw"] * np.sum((u - T_w)) * dA  # W
    Q_loss = params["UA_loss"] * (T_w - T_amb)      # W

    # Headspace
    Vg = headspace_volume(m_w)
    Tg = T_w + 273.15
    Psat = psat_water_pa(T_w)
    Pv = (nv * R_gas * Tg) / Vg

    # Condensation clamp
    if Pv > Psat:
        nv = Psat * Vg / (R_gas * Tg)
        Pv = Psat

    # Evaporation
    if params["lid_on"]:
        dn_evap = params["K_evap"] * A_surf * max(Psat - Pv, 0.0)  # mol/s
    else:
        Pv_amb = params["RH_amb"] * psat_water_pa(T_amb)
        dn_evap = params["K_evap"] * params["open_evap_mult"] * A_surf * max(Psat - Pv_amb, 0.0)

    mdot_evap = dn_evap * M_w  # kg/s

    # Water energy
    denom = max(m_w * cp_w, 1e-9)
    dTdt = (Q_in - mdot_evap * h_fg - Q_loss) / denom
    T_w = max(T_w + dTdt * dt, -10.0)

    # Water mass update
    m_w = max(m_w - mdot_evap * dt, 0.0)

    # Vapor moles from evaporation
    nv = max(nv + dn_evap * dt, 0.0)

    # Venting (lid off => relax total moles toward ambient)
    if not params["lid_on"]:
        Vg = headspace_volume(m_w)
        Tg = T_w + 273.15
        n_tot = na + nv
        n_eq = P_amb * Vg / (R_gas * Tg)
        dn_tot_dt = -(n_tot - n_eq) / max(params["tau_vent"], 1e-6)
        dn_tot = dn_tot_dt * dt

        frac_v = (nv / n_tot) if n_tot > 1e-12 else 0.0
        nv = max(nv + frac_v * dn_tot, 0.0)
        na = max(na + (1.0 - frac_v) * dn_tot, 0.0)

        # re-clamp saturation
        Psat = psat_water_pa(T_w)
        Vg = headspace_volume(m_w)
        Tg = T_w + 273.15
        Pv = (nv * R_gas * Tg) / Vg
        if Pv > Psat:
            nv = Psat * Vg / (R_gas * Tg)

    # Pressures for display
    Vg = headspace_volume(m_w)
    Tg = T_w + 273.15
    P = (na + nv) * R_gas * Tg / Vg
    Pv = (nv * R_gas * Tg) / Vg

    return u, T_w, m_w, nv, na, P, Pv, mdot_evap, T_burner

# ---------------------------
# F) Interactive UI setup
# ---------------------------
plt.ion()
fig = plt.figure(figsize=(13, 6))

ax_map = fig.add_axes([0.05, 0.18, 0.42, 0.75])
ax_ts  = fig.add_axes([0.52, 0.18, 0.45, 0.75])

# Slider panel (bottom)
ax_hmw   = fig.add_axes([0.07, 0.11, 0.35, 0.03])
ax_Tburn = fig.add_axes([0.07, 0.07, 0.35, 0.03])
ax_UA    = fig.add_axes([0.07, 0.03, 0.35, 0.03])

ax_Ke    = fig.add_axes([0.55, 0.11, 0.35, 0.03])
ax_tau   = fig.add_axes([0.55, 0.07, 0.35, 0.03])
ax_RH    = fig.add_axes([0.55, 0.03, 0.35, 0.03])

# Buttons
ax_power = fig.add_axes([0.44, 0.11, 0.07, 0.06])
ax_lid   = fig.add_axes([0.44, 0.03, 0.07, 0.06])
ax_reset = fig.add_axes([0.44, 0.07, 0.07, 0.035])

# Initial plot objects
im = ax_map.imshow(u, cmap="hot", vmin=20, vmax=220, extent=[0, L, 0, L])
plt.colorbar(im, ax=ax_map, label="Pot-bottom Temp (°C)")
ax_map.set_title("2D Pot-Bottom Temperature")

t_hist = []
Tw_hist = []
P_hist = []
mdot_hist = []

ln_Tw, = ax_ts.plot([], [], label="Water Temp (°C)")
ln_dP, = ax_ts.plot([], [], label="Headspace ΔP (kPa)")
ln_md, = ax_ts.plot([], [], label="Evap m_dot (g/s)")
ax_ts.grid(True)
ax_ts.set_xlabel("Time (s)")
ax_ts.legend(loc="upper left")

# Sliders ("twist knobs")
s_hmw = Slider(ax_hmw, "h_mw (W/m²K)", 200.0, 15000.0, valinit=state["h_mw"])
s_Tb  = Slider(ax_Tburn, "Burner T (°C)", 60.0, 300.0, valinit=state["T_burner_on"])
s_UA  = Slider(ax_UA, "UA_loss (W/K)", 0.0, 60.0, valinit=state["UA_loss"])

s_Ke  = Slider(ax_Ke, "K_evap (×1e-7)", 0.0, 20.0, valinit=state["K_evap"]/1e-7)
s_tau = Slider(ax_tau, "tau_vent (s)", 0.05, 2.0, valinit=state["tau_vent"])
s_RH  = Slider(ax_RH, "RH_amb", 0.0, 1.0, valinit=state["RH_amb"])

# Buttons
b_power = Button(ax_power, "POWER: ON")
b_lid   = Button(ax_lid,   "LID: ON")
b_reset = Button(ax_reset, "RESET")

# Simulation state
sim = {
    "t": 0.0,
    "u": u.copy(),
    "T_w": T_w0,
    "m_w": m_w0,
    "nv": nv0,
    "na": na0,
    "running": True,
}

def refresh_ts_axes():
    ax_ts.relim()
    ax_ts.autoscale_view()

def update_from_sliders(_=None):
    state["h_mw"] = s_hmw.val
    state["T_burner_on"] = s_Tb.val
    state["UA_loss"] = s_UA.val

    state["K_evap"] = s_Ke.val * 1e-7
    state["tau_vent"] = s_tau.val
    state["RH_amb"] = s_RH.val

for s in (s_hmw, s_Tb, s_UA, s_Ke, s_tau, s_RH):
    s.on_changed(update_from_sliders)

def on_power(_):
    state["power_on"] = not state["power_on"]
    b_power.label.set_text(f"POWER: {'ON' if state['power_on'] else 'OFF'}")

def on_lid(_):
    state["lid_on"] = not state["lid_on"]
    b_lid.label.set_text(f"LID: {'ON' if state['lid_on'] else 'OFF'}")

def on_reset(_):
    # reset sim state and histories
    sim["t"] = 0.0
    sim["u"] = np.full((N, N), T_amb)
    sim["T_w"] = T_w0
    sim["m_w"] = m_w0
    sim["nv"] = nv0
    sim["na"] = na0

    t_hist.clear()
    Tw_hist.clear()
    P_hist.clear()
    mdot_hist.clear()

    # refresh plots immediately
    im.set_data(sim["u"])
    ln_Tw.set_data([], [])
    ln_dP.set_data([], [])
    ln_md.set_data([], [])
    ax_map.set_title("2D Pot-Bottom Temperature (reset)")
    ax_ts.set_title("")
    fig.canvas.draw_idle()

b_power.on_clicked(on_power)
b_lid.on_clicked(on_lid)
b_reset.on_clicked(on_reset)

# ---------------------------
# G) Interactive loop
# ---------------------------
print("Close the figure window to stop.")
update_from_sliders()

try:
    while plt.fignum_exists(fig.number):
        # step a few times per UI refresh so it feels smooth
        for _ in range(10):
            sim["u"], sim["T_w"], sim["m_w"], sim["nv"], sim["na"], P, Pv, mdot, Tb = vertical_step(
                sim["u"], sim["T_w"], sim["m_w"], sim["nv"], sim["na"], state
            )
            sim["t"] += dt

            t_hist.append(sim["t"])
            Tw_hist.append(sim["T_w"])
            P_hist.append(P)
            mdot_hist.append(mdot)

        # update visuals
        im.set_data(sim["u"])
        ax_map.set_title(
            f"2D Pot-Bottom | t={sim['t']:.1f}s | POWER={'ON' if state['power_on'] else 'OFF'} | LID={'ON' if state['lid_on'] else 'OFF'} | Tb={Tb:.0f}°C"
        )

        ln_Tw.set_data(t_hist, Tw_hist)
        ln_dP.set_data(t_hist, (np.array(P_hist) - P_amb)/1000.0)     # kPa
        ln_md.set_data(t_hist, np.array(mdot_hist)*1000.0)            # g/s

        refresh_ts_axes()
        fig.canvas.draw_idle()
        plt.pause(0.001)

except KeyboardInterrupt:
    pass

plt.ioff()
plt.show()

2D explicit stability parameter r = 0.0038 (want <= ~0.25)
Close the figure window to stop.


Traceback (most recent call last):
  File "C:\Users\Sam\AppData\Local\Python\pythoncore-3.14-64\Lib\site-packages\matplotlib\cbook.py", line 361, in process
    func(*args, **kwargs)
    ~~~~^^^^^^^^^^^^^^^^^
  File "C:\Users\Sam\AppData\Local\Python\pythoncore-3.14-64\Lib\site-packages\matplotlib\widgets.py", line 216, in _click
    event.canvas.grab_mouse(self.ax)
    ~~~~~~~~~~~~~~~~~~~~~~~^^^^^^^^^
  File "C:\Users\Sam\AppData\Local\Python\pythoncore-3.14-64\Lib\site-packages\matplotlib\backend_bases.py", line 1837, in grab_mouse
    raise RuntimeError("Another Axes already grabs mouse input")
RuntimeError: Another Axes already grabs mouse input


In [20]:
import numpy as np
import matplotlib.pyplot as plt
from matplotlib.widgets import Slider, Button


%matplotlib tk
# ============================================================
# Interactive: 2D pot-bottom heat diffusion + 1D vertical pot model
# Sliders ("twist knobs") + Power button
# ============================================================



# ---------------------------
# A) Geometry / Numerics
# ---------------------------
L = 0.20
N = 60
dx = L / N
dt = 0.01

alpha_steel = 4.2e-6  # m^2/s
r = alpha_steel * dt / dx**2
print(f"2D explicit stability parameter r = {r:.4f} (want <= ~0.25)")

center = N // 2
radius = N // 4
Y, X = np.ogrid[:N, :N]
burner = (X - center)**2 + (Y - center)**2 <= radius**2

def apply_neumann_edges(arr):
    arr[0, :]  = arr[1, :]
    arr[-1, :] = arr[-2, :]
    arr[:, 0]  = arr[:, 1]
    arr[:, -1] = arr[:, -2]
    return arr

def step_metal(u, T_burner):
    u_next = u.copy()
    lap = (
        u[2:, 1:-1] + u[:-2, 1:-1] +
        u[1:-1, 2:] + u[1:-1, :-2] -
        4.0 * u[1:-1, 1:-1]
    )
    u_next[1:-1, 1:-1] = u[1:-1, 1:-1] + r * lap
    u_next = apply_neumann_edges(u_next)
    u_next[burner] = T_burner
    return u_next

# ---------------------------
# B) Pot / Water / Headspace model
# ---------------------------
R_pot = 0.10
H_pot = 0.12
A_surf = np.pi * R_pot**2

fill_frac = 1/3
rho_w = 997.0
cp_w = 4180.0
T_amb = 20.0
P_amb = 101325.0
R_gas = 8.314462618
M_w = 0.01801528
h_fg = 2.256e6

def psat_water_pa(Tc):
    # Antoine (1–100C). Still behaves reasonably outside, but treat as approximate.
    A = 8.07131
    B = 1730.63
    C = 233.426
    P_mmHg = 10 ** (A - B / (C + Tc))
    return P_mmHg * 133.322368

def headspace_volume(m_w):
    V_w = m_w / rho_w
    V_total = A_surf * H_pot
    return max(V_total - V_w, 1e-6)

# ---------------------------
# C) Initial state
# ---------------------------
u = np.full((N, N), T_amb)

V_w0 = A_surf * (H_pot * fill_frac)
m_w0 = rho_w * V_w0
T_w0 = T_amb

RH0 = 0.50
Vg0 = headspace_volume(m_w0)
Tg0 = T_w0 + 273.15
Pv0 = RH0 * psat_water_pa(T_w0)
nv0 = Pv0 * Vg0 / (R_gas * Tg0)
na0 = (P_amb - Pv0) * Vg0 / (R_gas * Tg0)

# ---------------------------
# D) Controls (defaults)
# ---------------------------
state = {
    "power_on": True,
    "lid_on": True,
    "T_burner_on": 180.0,   # °C when power is ON
    "T_burner_off": T_amb,  # °C when power is OFF
    "h_mw": 2500.0,         # W/m^2/K metal->water
    "K_evap": 2e-7,         # mol/(s*m^2*Pa)
    "tau_vent": 0.35,       # s
    "UA_loss": 8.0,         # W/K
    "open_evap_mult": 8.0,  # faster evaporation when lid OFF
    "RH_amb": RH0,          # ambient RH used when lid OFF
}

# ---------------------------
# E) Simulation step for vertical model
# ---------------------------
def vertical_step(u, T_w, m_w, nv, na, params):
    # Burner temp based on power
    T_burner = params["T_burner_on"] if params["power_on"] else params["T_burner_off"]

    # Step the metal
    u = step_metal(u, T_burner)

    # Heat input from metal to water
    dA = (L / N) ** 2
    Q_in = params["h_mw"] * np.sum((u - T_w)) * dA  # W
    Q_loss = params["UA_loss"] * (T_w - T_amb)      # W

    # Headspace
    Vg = headspace_volume(m_w)
    Tg = T_w + 273.15
    Psat = psat_water_pa(T_w)
    Pv = (nv * R_gas * Tg) / Vg

    # Condensation clamp
    if Pv > Psat:
        nv = Psat * Vg / (R_gas * Tg)
        Pv = Psat

    # Evaporation
    if params["lid_on"]:
        dn_evap = params["K_evap"] * A_surf * max(Psat - Pv, 0.0)  # mol/s
    else:
        Pv_amb = params["RH_amb"] * psat_water_pa(T_amb)
        dn_evap = params["K_evap"] * params["open_evap_mult"] * A_surf * max(Psat - Pv_amb, 0.0)

    mdot_evap = dn_evap * M_w  # kg/s

    # Water energy
    denom = max(m_w * cp_w, 1e-9)
    dTdt = (Q_in - mdot_evap * h_fg - Q_loss) / denom
    T_w = max(T_w + dTdt * dt, -10.0)

    # Water mass update
    m_w = max(m_w - mdot_evap * dt, 0.0)

    # Vapor moles from evaporation
    nv = max(nv + dn_evap * dt, 0.0)

    # Venting (lid off => relax total moles toward ambient)
    if not params["lid_on"]:
        Vg = headspace_volume(m_w)
        Tg = T_w + 273.15
        n_tot = na + nv
        n_eq = P_amb * Vg / (R_gas * Tg)
        dn_tot_dt = -(n_tot - n_eq) / max(params["tau_vent"], 1e-6)
        dn_tot = dn_tot_dt * dt

        frac_v = (nv / n_tot) if n_tot > 1e-12 else 0.0
        nv = max(nv + frac_v * dn_tot, 0.0)
        na = max(na + (1.0 - frac_v) * dn_tot, 0.0)

        # re-clamp saturation
        Psat = psat_water_pa(T_w)
        Vg = headspace_volume(m_w)
        Tg = T_w + 273.15
        Pv = (nv * R_gas * Tg) / Vg
        if Pv > Psat:
            nv = Psat * Vg / (R_gas * Tg)

    # Pressures for display
    Vg = headspace_volume(m_w)
    Tg = T_w + 273.15
    P = (na + nv) * R_gas * Tg / Vg
    Pv = (nv * R_gas * Tg) / Vg

    return u, T_w, m_w, nv, na, P, Pv, mdot_evap, T_burner

# ---------------------------
# F) Interactive UI setup
# ---------------------------
plt.ion()
fig = plt.figure(figsize=(13, 6))

ax_map = fig.add_axes([0.05, 0.18, 0.42, 0.75])
ax_ts  = fig.add_axes([0.52, 0.18, 0.45, 0.75])

# Slider panel (bottom)
ax_hmw   = fig.add_axes([0.07, 0.11, 0.35, 0.03])
ax_Tburn = fig.add_axes([0.07, 0.07, 0.35, 0.03])
ax_UA    = fig.add_axes([0.07, 0.03, 0.35, 0.03])

ax_Ke    = fig.add_axes([0.55, 0.11, 0.35, 0.03])
ax_tau   = fig.add_axes([0.55, 0.07, 0.35, 0.03])
ax_RH    = fig.add_axes([0.55, 0.03, 0.35, 0.03])

# Buttons
ax_power = fig.add_axes([0.44, 0.11, 0.07, 0.06])
ax_lid   = fig.add_axes([0.44, 0.03, 0.07, 0.06])
ax_reset = fig.add_axes([0.44, 0.07, 0.07, 0.035])

# Initial plot objects
im = ax_map.imshow(u, cmap="hot", vmin=20, vmax=220, extent=[0, L, 0, L])
plt.colorbar(im, ax=ax_map, label="Pot-bottom Temp (°C)")
ax_map.set_title("2D Pot-Bottom Temperature")

t_hist = []
Tw_hist = []
P_hist = []
mdot_hist = []

ln_Tw, = ax_ts.plot([], [], label="Water Temp (°C)")
ln_dP, = ax_ts.plot([], [], label="Headspace ΔP (kPa)")
ln_md, = ax_ts.plot([], [], label="Evap m_dot (g/s)")
ax_ts.grid(True)
ax_ts.set_xlabel("Time (s)")
ax_ts.legend(loc="upper left")

# Sliders ("twist knobs")
s_hmw = Slider(ax_hmw, "h_mw (W/m²K)", 200.0, 15000.0, valinit=state["h_mw"])
s_Tb  = Slider(ax_Tburn, "Burner T (°C)", 60.0, 300.0, valinit=state["T_burner_on"])
s_UA  = Slider(ax_UA, "UA_loss (W/K)", 0.0, 60.0, valinit=state["UA_loss"])

s_Ke  = Slider(ax_Ke, "K_evap (×1e-7)", 0.0, 20.0, valinit=state["K_evap"]/1e-7)
s_tau = Slider(ax_tau, "tau_vent (s)", 0.05, 2.0, valinit=state["tau_vent"])
s_RH  = Slider(ax_RH, "RH_amb", 0.0, 1.0, valinit=state["RH_amb"])

# Buttons
b_power = Button(ax_power, "POWER: ON")
b_lid   = Button(ax_lid,   "LID: ON")
b_reset = Button(ax_reset, "RESET")

# Simulation state
sim = {
    "t": 0.0,
    "u": u.copy(),
    "T_w": T_w0,
    "m_w": m_w0,
    "nv": nv0,
    "na": na0,
    "running": True,
}

def refresh_ts_axes():
    ax_ts.relim()
    ax_ts.autoscale_view()

def update_from_sliders(_=None):
    state["h_mw"] = s_hmw.val
    state["T_burner_on"] = s_Tb.val
    state["UA_loss"] = s_UA.val

    state["K_evap"] = s_Ke.val * 1e-7
    state["tau_vent"] = s_tau.val
    state["RH_amb"] = s_RH.val

for s in (s_hmw, s_Tb, s_UA, s_Ke, s_tau, s_RH):
    s.on_changed(update_from_sliders)

def on_power(_):
    state["power_on"] = not state["power_on"]
    b_power.label.set_text(f"POWER: {'ON' if state['power_on'] else 'OFF'}")

def on_lid(_):
    state["lid_on"] = not state["lid_on"]
    b_lid.label.set_text(f"LID: {'ON' if state['lid_on'] else 'OFF'}")

def on_reset(_):
    # reset sim state and histories
    sim["t"] = 0.0
    sim["u"] = np.full((N, N), T_amb)
    sim["T_w"] = T_w0
    sim["m_w"] = m_w0
    sim["nv"] = nv0
    sim["na"] = na0

    t_hist.clear()
    Tw_hist.clear()
    P_hist.clear()
    mdot_hist.clear()

    # refresh plots immediately
    im.set_data(sim["u"])
    ln_Tw.set_data([], [])
    ln_dP.set_data([], [])
    ln_md.set_data([], [])
    ax_map.set_title("2D Pot-Bottom Temperature (reset)")
    ax_ts.set_title("")
    fig.canvas.draw_idle()

b_power.on_clicked(on_power)
b_lid.on_clicked(on_lid)
b_reset.on_clicked(on_reset)

# ---------------------------
# G) Timer-based simulation loop (FIXED VERSION)
# ---------------------------

def timer_update(event=None):
    # step simulation multiple times per frame for smoothness
    for _ in range(10):
        sim["u"], sim["T_w"], sim["m_w"], sim["nv"], sim["na"], P, Pv, mdot, Tb = vertical_step(
            sim["u"], sim["T_w"], sim["m_w"], sim["nv"], sim["na"], state
        )
        sim["t"] += dt

        t_hist.append(sim["t"])
        Tw_hist.append(sim["T_w"])
        P_hist.append(P)
        mdot_hist.append(mdot)

    # update visuals
    im.set_data(sim["u"])
    ax_map.set_title(
        f"2D Pot-Bottom | t={sim['t']:.1f}s | "
        f"POWER={'ON' if state['power_on'] else 'OFF'} | "
        f"LID={'ON' if state['lid_on'] else 'OFF'} | "
        f"Tb={Tb:.0f}°C"
    )

    ln_Tw.set_data(t_hist, Tw_hist)
    ln_dP.set_data(t_hist, (np.array(P_hist) - P_amb)/1000.0)
    ln_md.set_data(t_hist, np.array(mdot_hist)*1000.0)

    ax_ts.relim()
    ax_ts.autoscale_view()

    fig.canvas.draw_idle()

# Create timer
timer = fig.canvas.new_timer(interval=20)  # ms
timer.add_callback(timer_update)
timer.start()

plt.show()

2D explicit stability parameter r = 0.0038 (want <= ~0.25)


In [22]:
# this is a summary of the variables

# ---------------------------
# A) Geometry / Numerics (2D Metal Plate Model)
# ---------------------------

L = 0.20          # [m] Physical width of modeled pot bottom (20 cm diameter footprint)
N = 60            # [-] Number of grid cells per side (NxN resolution)
dx = L / N        # [m] Spatial step size (grid cell width)

dt = 0.01         # [s] Time step for simulation updates

alpha_steel = 4.2e-6  
# [m^2/s] Thermal diffusivity of stainless steel
# Controls how fast heat spreads across pot bottom

r = alpha_steel * dt / dx**2  
# [-] Explicit diffusion stability parameter (must be <= ~0.25 for stability)

center = N // 2   # Grid index of burner center
radius = N // 4   # Burner radius in grid cells

Y, X = np.ogrid[:N, :N]
burner = (X - center)**2 + (Y - center)**2 <= radius**2
# Boolean mask identifying burner region


# ---------------------------
# B) Pot / Water / Headspace Model
# ---------------------------

R_pot = 0.10      
# [m] Inner pot radius (~10 cm)

H_pot = 0.12      
# [m] Inner pot height (~12 cm)

A_surf = np.pi * R_pot**2  
# [m^2] Cross-sectional area of pot
# Also surface area of water

fill_frac = 1/3   
# [-] Initial fraction of pot height filled with water

rho_w = 997.0     
# [kg/m^3] Density of liquid water (near room temp)

cp_w = 4180.0     
# [J/(kg*K)] Specific heat capacity of water

T_amb = 20.0      
# [°C] Ambient room temperature

P_amb = 101325.0  
# [Pa] Ambient atmospheric pressure

R_gas = 8.314462618  
# [J/(mol*K)] Universal gas constant

M_w = 0.01801528  
# [kg/mol] Molecular weight of water vapor

h_fg = 2.256e6    
# [J/kg] Latent heat of vaporization of water (~100C value)



# ---------------------------
# C) Initial State Variables
# ---------------------------

V_w0 = A_surf * (H_pot * fill_frac)
# [m^3] Initial water volume

m_w0 = rho_w * V_w0
# [kg] Initial water mass

T_w0 = T_amb
# [°C] Initial bulk water temperature

RH0 = 0.50
# [-] Initial relative humidity inside headspace (50%)

Vg0 = headspace_volume(m_w0)
# [m^3] Initial gas (headspace) volume

Tg0 = T_w0 + 273.15
# [K] Initial gas temperature (assumed equal to water)

Pv0 = RH0 * psat_water_pa(T_w0)
# [Pa] Initial vapor partial pressure

nv0 = Pv0 * Vg0 / (R_gas * Tg0)
# [mol] Initial moles of water vapor in headspace

na0 = (P_amb - Pv0) * Vg0 / (R_gas * Tg0)
# [mol] Initial moles of dry air trapped in headspace


# ---------------------------
# D) Adjustable Physical Parameters (Interactive Controls)
# ---------------------------

state = {

    "power_on": True,
    # Boolean: Stove heating active or not

    "lid_on": True,
    # Boolean: Lid sealed (True) or venting/open (False)

    "T_burner_on": 180.0,
    # [°C] Burner region temperature when power is ON

    "T_burner_off": T_amb,
    # [°C] Burner temperature when power is OFF

    "h_mw": 2500.0,
    # [W/(m^2*K)] Heat transfer coefficient from metal bottom to water
    # Controls how efficiently heat enters water

    "K_evap": 2e-7,
    # [mol/(s*m^2*Pa)] Evaporation kinetic coefficient
    # Controls how fast vapor pressure approaches saturation

    "tau_vent": 0.35,
    # [s] Time constant for headspace venting when lid is off
    # Smaller = more violent pressure equalization

    "UA_loss": 8.0,
    # [W/K] Lumped heat loss from water to ambient
    # Represents conduction through walls + convection to air

    "open_evap_mult": 8.0,
    # [-] Multiplier for evaporation rate when lid is open

    "RH_amb": RH0,
    # [-] Ambient relative humidity (used when lid open)
}



sim = {
    "t": 0.0,      # [s] Simulation time
    "u": u.copy(), # [°C] 2D metal temperature field
    "T_w": T_w0,   # [°C] Bulk water temperature
    "m_w": m_w0,   # [kg] Remaining liquid water mass
    "nv": nv0,     # [mol] Water vapor moles in headspace
    "na": na0,     # [mol] Dry air moles in headspace
}

Physical Meaning Summary (So You Always Remember)
2D Layer

u → temperature field of pot bottom

alpha_steel → how fast heat spreads laterally

Vertical / Bulk Layer

T_w → bulk water temperature

m_w → water mass (shrinks due to evaporation)

Gas Layer

nv → vapor molecules

na → trapped air molecules

P → total pressure via ideal gas law

Phase Change

K_evap → how aggressively vapor pressure tries to reach saturation

h_fg → energy penalty for evaporation

Vent Physics

tau_vent → how fast pressure equalizes when lid open

🧠 Why This Matters

Now the code reads like a thermodynamics textbook:

Energy balance

Phase equilibrium

Ideal gas behavior

Mass conservation

Transient diffusion

This is no longer “just a script.”
It’s a structured multiphysics model.

Parallel Programming version, may need agood pc to do this.

In [2]:
import numpy as np
import matplotlib.pyplot as plt
from matplotlib.widgets import Slider, Button
%matplotlib tk
# Optional speedup
try:
    from numba import njit, prange
    NUMBA_OK = True
except Exception:
    NUMBA_OK = False
    njit = None
    prange = range

# ============================================================
# Interactive: 2D pot-bottom heat diffusion + 1D vertical pot model
# Optimized: Numba-compiled stepping + timer-based GUI updates
# ============================================================

# ---------------------------
# A) Geometry / Numerics (2D metal plate)
# ---------------------------
L = 0.20          # [m] modeled pot bottom width (20 cm)
N = 120           # [-] grid resolution (try 60–200)
dx = L / N        # [m] spatial step
dt = 0.01         # [s] timestep

alpha_steel = 4.2e-6  # [m^2/s] thermal diffusivity
r = alpha_steel * dt / dx**2  # stability parameter
print(f"2D explicit stability r = {r:.4f} (want <= ~0.25)")

# Burner mask (circle at center)
center = N // 2
radius = N // 4
Y, X = np.ogrid[:N, :N]
burner_mask = ((X - center)**2 + (Y - center)**2 <= radius**2).astype(np.uint8)  # uint8 for numba

# ---------------------------
# B) Pot / Water / Headspace parameters (1D "vertical" model)
# ---------------------------
R_pot = 0.10             # [m] pot radius
H_pot = 0.12             # [m] pot height
A_surf = np.pi * R_pot**2  # [m^2] water surface area

fill_frac = 1/3
rho_w = 997.0            # [kg/m^3] water density
cp_w = 4180.0            # [J/(kg*K)] water heat capacity

T_amb = 20.0             # [°C]
P_amb = 101325.0         # [Pa]
R_gas = 8.314462618      # [J/(mol*K)]
M_w = 0.01801528         # [kg/mol]
h_fg = 2.256e6           # [J/kg] latent heat of vaporization (approx)

# ---------------------------
# C) Helper functions
# ---------------------------
def headspace_volume(m_w):
    """[m^3] Headspace volume = pot total volume - water volume."""
    V_w = m_w / rho_w
    V_total = A_surf * H_pot
    return max(V_total - V_w, 1e-9)

def psat_water_pa_py(Tc):
    """
    Saturation vapor pressure of water [Pa].
    Antoine equation ~valid ~1–100C (still behaves outside; treat as approx).
    """
    A = 8.07131
    B = 1730.63
    C = 233.426
    P_mmHg = 10 ** (A - B / (C + Tc))
    return P_mmHg * 133.322368

# We’ll implement psat in numba too (needs pure numeric ops)
if NUMBA_OK:
    @njit
    def psat_water_pa(Tc):
        A = 8.07131
        B = 1730.63
        C = 233.426
        P_mmHg = 10.0 ** (A - B / (C + Tc))
        return P_mmHg * 133.322368
else:
    psat_water_pa = psat_water_pa_py

# ---------------------------
# D) Numba-accelerated hot path
# ---------------------------
if NUMBA_OK:
    @njit(parallel=True, fastmath=True)
    def step_metal_numba(u, u_next, burner_mask, T_burner, r):
        """
        One explicit diffusion step on the 2D metal plate with:
        - insulated (Neumann) edges (zero-gradient)
        - fixed burner region temperature (Dirichlet)
        """
        n = u.shape[0]

        # interior diffusion (parallel)
        for i in prange(1, n - 1):
            for j in range(1, n - 1):
                lap = (u[i+1, j] + u[i-1, j] + u[i, j+1] + u[i, j-1] - 4.0 * u[i, j])
                u_next[i, j] = u[i, j] + r * lap

        # Neumann edges (copy adjacent interior)
        for j in prange(n):
            u_next[0, j] = u_next[1, j]
            u_next[n-1, j] = u_next[n-2, j]
        for i in prange(n):
            u_next[i, 0] = u_next[i, 1]
            u_next[i, n-1] = u_next[i, n-2]

        # enforce burner
        for i in prange(n):
            for j in range(n):
                if burner_mask[i, j] == 1:
                    u_next[i, j] = T_burner

    @njit(parallel=True, fastmath=True)
    def q_in_numba(u, T_w, h_mw, dA):
        """Compute Q_in = h_mw * sum((u - T_w) * dA)."""
        n = u.shape[0]
        acc = 0.0
        for i in prange(n):
            row_sum = 0.0
            for j in range(n):
                row_sum += (u[i, j] - T_w)
            acc += row_sum
        return h_mw * acc * dA

    @njit(fastmath=True)
    def vertical_step_numba(u, u_next, burner_mask,
                            T_w, m_w, nv, na,
                            power_on, lid_on,
                            T_burner_on, T_burner_off,
                            h_mw, UA_loss, K_evap, tau_vent,
                            open_evap_mult, RH_amb,
                            r, dA):
        """
        Advance:
        - 2D metal field u -> u_next
        - 1D water bulk temp T_w, water mass m_w
        - headspace moles nv, na
        Returns updated (u_next, T_w, m_w, nv, na, P, Pv, mdot, Tb)
        """

        # burner temperature based on power
        Tb = T_burner_on if power_on else T_burner_off

        # step metal plate
        step_metal_numba(u, u_next, burner_mask, Tb, r)

        # heat input from metal to water
        Q_in = q_in_numba(u_next, T_w, h_mw, dA)
        Q_loss = UA_loss * (T_w - T_amb)

        # headspace geometry
        V_w = m_w / rho_w
        V_total = A_surf * H_pot
        Vg = V_total - V_w
        if Vg < 1e-9:
            Vg = 1e-9

        Tg = T_w + 273.15
        Psat = psat_water_pa(T_w)
        Pv = (nv * R_gas * Tg) / Vg

        # condensation clamp
        if Pv > Psat:
            nv = Psat * Vg / (R_gas * Tg)
            Pv = Psat

        # evaporation kinetics
        if lid_on:
            drive = Psat - Pv
            if drive < 0.0:
                drive = 0.0
            dn_evap = K_evap * A_surf * drive
        else:
            Pv_amb = RH_amb * psat_water_pa(T_amb)
            drive = Psat - Pv_amb
            if drive < 0.0:
                drive = 0.0
            dn_evap = K_evap * open_evap_mult * A_surf * drive

        mdot = dn_evap * M_w  # kg/s

        # water energy balance
        denom = m_w * cp_w
        if denom < 1e-9:
            denom = 1e-9
        dTdt = (Q_in - mdot * h_fg - Q_loss) / denom
        T_w = T_w + dTdt * dt
        if T_w < -10.0:
            T_w = -10.0

        # water mass
        m_w = m_w - mdot * dt
        if m_w < 0.0:
            m_w = 0.0

        # vapor moles increase
        nv = nv + dn_evap * dt
        if nv < 0.0:
            nv = 0.0

        # venting when lid off: relax total moles toward ambient
        if not lid_on:
            # recompute Vg, Tg after updates
            V_w = m_w / rho_w
            Vg = (A_surf * H_pot) - V_w
            if Vg < 1e-9:
                Vg = 1e-9
            Tg = T_w + 273.15

            n_tot = na + nv
            n_eq = P_amb * Vg / (R_gas * Tg)
            if tau_vent < 1e-6:
                tau_vent = 1e-6
            dn_tot_dt = -(n_tot - n_eq) / tau_vent
            dn_tot = dn_tot_dt * dt

            frac_v = 0.0
            if n_tot > 1e-12:
                frac_v = nv / n_tot

            nv = nv + frac_v * dn_tot
            na = na + (1.0 - frac_v) * dn_tot
            if nv < 0.0:
                nv = 0.0
            if na < 0.0:
                na = 0.0

            # saturation clamp again
            Psat = psat_water_pa(T_w)
            Pv = (nv * R_gas * Tg) / Vg
            if Pv > Psat:
                nv = Psat * Vg / (R_gas * Tg)

        # pressures for logging
        V_w = m_w / rho_w
        Vg = (A_surf * H_pot) - V_w
        if Vg < 1e-9:
            Vg = 1e-9
        Tg = T_w + 273.15
        P = (na + nv) * R_gas * Tg / Vg
        Pv = (nv * R_gas * Tg) / Vg

        return T_w, m_w, nv, na, P, Pv, mdot, Tb

# ---------------------------
# E) Initial conditions
# ---------------------------
u = np.full((N, N), T_amb, dtype=np.float64)
u_next = u.copy()

V_w0 = A_surf * (H_pot * fill_frac)
m_w0 = rho_w * V_w0
T_w0 = T_amb

RH0 = 0.50
Vg0 = headspace_volume(m_w0)
Tg0 = T_w0 + 273.15
Pv0 = RH0 * psat_water_pa_py(T_w0)
nv0 = Pv0 * Vg0 / (R_gas * Tg0)
na0 = (P_amb - Pv0) * Vg0 / (R_gas * Tg0)

# ---------------------------
# F) Interactive controls ("knobs")
# ---------------------------
state = {
    "power_on": True,
    "lid_on": True,
    "T_burner_on": 180.0,
    "T_burner_off": T_amb,
    "h_mw": 2500.0,
    "K_evap": 2e-7,
    "tau_vent": 0.35,
    "UA_loss": 8.0,
    "open_evap_mult": 8.0,
    "RH_amb": RH0,
}

# ---------------------------
# G) Matplotlib UI (timer-based)
# ---------------------------
plt.ion()
fig = plt.figure(figsize=(13, 6))

ax_map = fig.add_axes([0.05, 0.18, 0.42, 0.75])
ax_ts  = fig.add_axes([0.52, 0.18, 0.45, 0.75])

# sliders
ax_hmw   = fig.add_axes([0.07, 0.11, 0.35, 0.03])
ax_Tburn = fig.add_axes([0.07, 0.07, 0.35, 0.03])
ax_UA    = fig.add_axes([0.07, 0.03, 0.35, 0.03])

ax_Ke    = fig.add_axes([0.55, 0.11, 0.35, 0.03])
ax_tau   = fig.add_axes([0.55, 0.07, 0.35, 0.03])
ax_RH    = fig.add_axes([0.55, 0.03, 0.35, 0.03])

# buttons
ax_power = fig.add_axes([0.44, 0.11, 0.07, 0.06])
ax_lid   = fig.add_axes([0.44, 0.03, 0.07, 0.06])
ax_reset = fig.add_axes([0.44, 0.07, 0.07, 0.035])

im = ax_map.imshow(u, cmap="hot", vmin=20, vmax=260, extent=[0, L, 0, L])
plt.colorbar(im, ax=ax_map, label="Pot-bottom Temp (°C)")
ax_map.set_title("2D Pot-Bottom Temperature")

t_hist, Tw_hist, P_hist, mdot_hist = [], [], [], []
ln_Tw, = ax_ts.plot([], [], label="Water Temp (°C)")
ln_dP, = ax_ts.plot([], [], label="Headspace ΔP (kPa)")
ln_md, = ax_ts.plot([], [], label="Evap m_dot (g/s)")
ax_ts.grid(True)
ax_ts.set_xlabel("Time (s)")
ax_ts.legend(loc="upper left")

s_hmw = Slider(ax_hmw, "h_mw (W/m²K)", 200.0, 15000.0, valinit=state["h_mw"])
s_Tb  = Slider(ax_Tburn, "Burner T (°C)", 60.0, 350.0, valinit=state["T_burner_on"])
s_UA  = Slider(ax_UA, "UA_loss (W/K)", 0.0, 80.0, valinit=state["UA_loss"])

s_Ke  = Slider(ax_Ke, "K_evap (×1e-7)", 0.0, 40.0, valinit=state["K_evap"] / 1e-7)
s_tau = Slider(ax_tau, "tau_vent (s)", 0.05, 2.0, valinit=state["tau_vent"])
s_RH  = Slider(ax_RH, "RH_amb", 0.0, 1.0, valinit=state["RH_amb"])

b_power = Button(ax_power, "POWER: ON")
b_lid   = Button(ax_lid,   "LID: ON")
b_reset = Button(ax_reset, "RESET")

# Simulation state
sim = {
    "t": 0.0,
    "T_w": T_w0,
    "m_w": m_w0,
    "nv": nv0,
    "na": na0,
}

# constant
dA = (L / N) ** 2

def update_from_sliders(_=None):
    state["h_mw"] = float(s_hmw.val)
    state["T_burner_on"] = float(s_Tb.val)
    state["UA_loss"] = float(s_UA.val)
    state["K_evap"] = float(s_Ke.val) * 1e-7
    state["tau_vent"] = float(s_tau.val)
    state["RH_amb"] = float(s_RH.val)

for s in (s_hmw, s_Tb, s_UA, s_Ke, s_tau, s_RH):
    s.on_changed(update_from_sliders)

def on_power(_):
    state["power_on"] = not state["power_on"]
    b_power.label.set_text(f"POWER: {'ON' if state['power_on'] else 'OFF'}")

def on_lid(_):
    state["lid_on"] = not state["lid_on"]
    b_lid.label.set_text(f"LID: {'ON' if state['lid_on'] else 'OFF'}")

def on_reset(_):
    sim["t"] = 0.0
    sim["T_w"] = T_w0
    sim["m_w"] = m_w0
    sim["nv"] = nv0
    sim["na"] = na0
    u[:, :] = T_amb
    u_next[:, :] = T_amb

    t_hist.clear(); Tw_hist.clear(); P_hist.clear(); mdot_hist.clear()
    im.set_data(u)
    ln_Tw.set_data([], [])
    ln_dP.set_data([], [])
    ln_md.set_data([], [])
    ax_map.set_title("2D Pot-Bottom Temperature (reset)")
    fig.canvas.draw_idle()

b_power.on_clicked(on_power)
b_lid.on_clicked(on_lid)
b_reset.on_clicked(on_reset)

# ---- warm-up compile (important: avoids first-click lag)
if NUMBA_OK:
    print("Numba enabled: warming up JIT compilation...")
    update_from_sliders()
    # do a couple steps to compile
    for _ in range(2):
        sim["T_w"], sim["m_w"], sim["nv"], sim["na"], P, Pv, mdot, Tb = vertical_step_numba(
            u, u_next, burner_mask,
            sim["T_w"], sim["m_w"], sim["nv"], sim["na"],
            state["power_on"], state["lid_on"],
            state["T_burner_on"], state["T_burner_off"],
            state["h_mw"], state["UA_loss"], state["K_evap"], state["tau_vent"],
            state["open_evap_mult"], state["RH_amb"],
            r, dA
        )
        u[:, :] = u_next
    print("JIT warm-up complete.")
else:
    print("Numba not available: running in pure Python/NumPy (slower).")

# ---- timer update
STEPS_PER_TICK = 25  # increase for faster physics per real-time second (CPU permitting)

def timer_update():
    update_from_sliders()

    # Step the simulation multiple times per GUI tick
    for _ in range(STEPS_PER_TICK):
        if NUMBA_OK:
            sim["T_w"], sim["m_w"], sim["nv"], sim["na"], P, Pv, mdot, Tb = vertical_step_numba(
                u, u_next, burner_mask,
                sim["T_w"], sim["m_w"], sim["nv"], sim["na"],
                state["power_on"], state["lid_on"],
                state["T_burner_on"], state["T_burner_off"],
                state["h_mw"], state["UA_loss"], state["K_evap"], state["tau_vent"],
                state["open_evap_mult"], state["RH_amb"],
                r, dA
            )
            # swap buffers
            u[:, :] = u_next
        else:
            # fallback: slower pure python version (minimal, uses numpy diffusion)
            # (kept brief—if you want, I can include a full non-numba fallback)
            pass

        sim["t"] += dt
        t_hist.append(sim["t"])
        Tw_hist.append(sim["T_w"])
        P_hist.append(P)
        mdot_hist.append(mdot)

    # Update visuals
    im.set_data(u)
    ax_map.set_title(
        f"2D Pot-Bottom | t={sim['t']:.1f}s | "
        f"POWER={'ON' if state['power_on'] else 'OFF'} | "
        f"LID={'ON' if state['lid_on'] else 'OFF'} | "
        f"Tb={Tb:.0f}°C | N={N}"
    )

    ln_Tw.set_data(t_hist, Tw_hist)
    ln_dP.set_data(t_hist, (np.array(P_hist) - P_amb) / 1000.0)   # kPa
    ln_md.set_data(t_hist, np.array(mdot_hist) * 1000.0)          # g/s

    ax_ts.relim()
    ax_ts.autoscale_view()
    fig.canvas.draw_idle()

timer = fig.canvas.new_timer(interval=20)  # ms
timer.add_callback(timer_update)
timer.start()

plt.show()

2D explicit stability r = 0.0151 (want <= ~0.25)
Numba enabled: warming up JIT compilation...
JIT warm-up complete.
